### Document Loading

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

directory_path = "../arch-wiki/html/en/"

if not os.path.exists(directory_path):
    print(f"Directory not found: {directory_path}")
else:
    loader = DirectoryLoader(
        path=directory_path,
        glob="**/*.html", # all html files and subdirectories
        loader_cls=TextLoader, # keeps html tags for document splitting
        show_progress=True
    )

docs = loader.load() # compare it to lazy load

/home/wtl04/coding/offline-sysadmin-assistant/venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
100%|████████████████████████████████████████████████████████████████████████████▉| 2484/2486 [00:00<00:00, 3760.35it/s]


In [2]:
len(docs)

2484

### HTMLSemanticPreservingSplitter

In [5]:
# BeautifulSoup is required to use the custom handlers
from bs4 import Tag
from langchain_text_splitters import HTMLSemanticPreservingSplitter

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

def code_handler(element: Tag) -> str:
    data_lang = element.get("data-lang")
    code_format = f"<code:{data_lang}>{element.get_text()}</code>"

    return code_format


splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    separators=["\n\n", "\n", ". ", "! ", "? "],
    max_chunk_size=1000,
    preserve_images=True,
    preserve_videos=True,
    elements_to_preserve=["table", "ul", "ol", "code"],
    denylist_tags=["script", "style", "head"],
    custom_handlers={"code": code_handler}, # define custom handlers for custom html tags
)

final_chunks=[]

for doc in docs:
    # if html file, split by header, then split again into smaller chunks
    if doc.metadata.get("source", "").endswith(".html"):
        documents = splitter.split_text(doc.page_content)
        final_chunks.extend(documents)
    else:
        # non-html, use recursive splitter directly
        final_chunks.extend(splitter.split_documents([doc]))

In [7]:
final_chunks

[Document(metadata={}, page_content='Jump to content'),
 Document(metadata={'Header 2': 'Contents'}, page_content='move to sidebar hide Beginning 1 Installation 2 Configuration Toggle Configuration subsection 2.1 /etc/shorewall/interfaces 2.2 /etc/shorewall/policy 2.3 /etc/shorewall/rules 2.4 /etc/shorewall/masq 2.4.1 SSH 2.4.2 Port forwarding (DNAT) 2.5 /etc/shorewall/stoppedrules 2.6 /etc/shorewall/shorewall.conf 3 Start 4 Traffic shaping Toggle the table of contents'),
 Document(metadata={'Header 1': 'Shorewall'}, page_content='Appearance move to sidebar hide From ArchWiki ![image:../File:Merge-arrows-2.svg](../File:Merge-arrows-2.svg) Iptables#Front-ends This article or section is a candidate for merging with . Notes: Talk:Shorewall Not improved since flagged. (Discuss in ) ![image:../File:Tango-view-fullscreen.svg](../File:Tango-view-fullscreen.svg) This article or section needs expansion. Reason: Talk:Shorewall This is mostly config dumps, there are no explanations why to configu

### Document Splitting

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

# initilaize header splitter
html_header_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# initilaize text splitter to ensure chunks fit in local llm
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

final_chunks=[]

for doc in docs:
    # if html file, split by header, then split again into smaller chunks
    if doc.metadata.get("source", "").endswith(".html"):
        header_splits = html_header_splitter.split_text(doc.page_content)

        # IMPORTANT: Manually carry over the 'source' from the original file
        for split in header_splits:
            split.metadata["source"] = doc.metadata.get("source")

        # DEBUG
        print(f"Found {len(header_splits)} header sections.")
        if len(header_splits) > 0:
            print(f"Metadata of first section: {header_splits[0].metadata}")

        
        final_chunks.extend(text_splitter.split_documents(header_splits))
    else:
        # non-html, use recursive splitter directly
        final_chunks.extend(text_splitter.split_documents([doc]))

Found 63 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/WireGuard.html'}
Found 15 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Sugar.html'}
Found 26 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Character_encoding.html'}
Found 13 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Category:Hardware_detection_and_troubleshooting.html'}
Found 27 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Intel_Quartus_Prime.html'}
Found 46 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Dhcpcd.html'}
Found 27 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Guitarix.html'}
Found 25 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Bluetooth_mouse.html'}
Found 25 header sections.
Metadata of first section: {'source': 'arch-wiki/html/en/Command-line_shell.html'}
Found 17 header sections.
Metadata o

In [10]:
len(final_chunks)

75185